In [2]:
from os import listdir
import os.path as op
import numpy as np

bids_folder = '/data/ds-stressrisk'

subList = ['01','02','03','04','05','09']
subList = [f[5:6] for f in listdir(bids_folder) if f[0:3] == 'sub']
ses = 1

target_folder = op.join(bids_folder,'derivatives','correlation_matrices')


In [4]:
from nilearn import datasets

atlas = datasets.fetch_atlas_surf_destrieux()
regions = atlas['labels'].copy()
masked_regions = [b'Medial_wall', b'Unknown']
masked_labels = [regions.index(r) for r in masked_regions] # [42, 0]

# Build Destrieux parcellation and mask
labeling = np.concatenate([atlas['map_left'], atlas['map_right']]) # atlas['map_left'] == atlas.map_left -> array, each vertex has a label assignment (a number from 0-51)
mask = ~np.isin(labeling, masked_labels)
N_vertices = len(np.where(mask==True)[0])

In [14]:
#matrix_zeros = np.zeros((20484, 20484))
matrix_zeros = np.zeros((N_vertices, N_vertices))
av_cm = matrix_zeros.copy()

for sub in subList:
    #correlation_matrix_resize = np.load(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_fsav5-format.npy'))
    correlation_matrix = np.load(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_fsav5_unfiltered.npy'))

    av_cm += np.arctan(correlation_matrix) # fisher-Z-transformed
    print(f'subject {sub} added')

av_cm = av_cm/len(subList)
av_cm_transf = np.tan(av_cm) # sanity check: diagonal should be 1 !

subject {sub} added
subject {sub} added
subject {sub} added
subject {sub} added


In [18]:
from brainspace.gradient import GradientMaps

gm = GradientMaps(n_components=2, random_state=0) # Default is 'dm' = DiffusionMaps
gm.fit(av_cm_transf)

file_name = f'cm_av{len(subList)}_unfiltered.npy'
np.save(op.join(target_folder,file_name),gm.gradients_)
print(f'saved to: {op.join(target_folder,file_name)}')

/home/ubuntu/miniconda3/envs/numrefields/lib/python3.10/site-packages/brainspace-0.1.4-py3.10.egg/brainspace/gradient/embedding.py:70: UserWarning: Affinity is not symmetric. Making symmetric.
  warnings.warn('Affinity is not symmetric. Making symmetric.')


GradientMaps(n_components=2, random_state=0)